In [1]:
!pip install -q medmnist
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

np.random.seed(42)
tf.random.set_seed(42)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Experiment 9: Transfer Learning using Pretrained CNN Models
**Aim:** To classify CIFAR-10 images using a pretrained MobileNetV2 (ImageNet weights) as a frozen feature extractor, followed by fine-tuning.

In [2]:
(x_tr, y_tr), (x_te, y_te) = tf.keras.datasets.cifar10.load_data()
x_tr, y_tr = x_tr[:10000], y_tr[:10000].ravel()
x_te, y_te = x_te[:2000], y_te[:2000].ravel()

img_size = 96
def prep(x, y):
    x = tf.image.resize(x, (img_size, img_size))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    return x, y

train_ds = tf.data.Dataset.from_tensor_slices((x_tr, y_tr)).map(prep).batch(64).prefetch(1)
test_ds = tf.data.Dataset.from_tensor_slices((x_te, y_te)).map(prep).batch(64).prefetch(1)

base = tf.keras.applications.MobileNetV2(input_shape=(img_size, img_size, 3), include_top=False, weights='imagenet')
base.trainable = False

tl_model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])
tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
tl_model.fit(train_ds, epochs=5, verbose=0)
print("Accuracy (frozen base):", tl_model.evaluate(test_ds, verbose=0)[1])

base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
tl_model.fit(train_ds, epochs=3, verbose=0)
print("Accuracy (after fine-tuning):", tl_model.evaluate(test_ds, verbose=0)[1])

C:\Users\Achintya\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Accuracy (frozen base): 0.8500000238418579


Accuracy (after fine-tuning): 0.8420000076293945
